# Check Free-Text Answers from the Fine-Tuned Blade Inspection Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/examples/wind-turbine-inspection/notebooks/free_text_check.ipynb)

The training notebooks teach [LFM3-VL-2B](https://huggingface.co/LiquidAI/LFM3-VL-2B) to answer with JSON boxes only. This notebook checks that the final model (SFT + GRPO adapter) still writes plain text: for a few held-out blade photos, it asks for a brief inspection finding with the defect type, the supporting evidence and the location, and shows each photo next to the answer. The base model's answer is shown for comparison.

There is no grading here: the dataset has no written inspection notes to score against. Use [`evaluate.ipynb`](evaluate.ipynb) for the measured results.

**Hardware:** any CUDA GPU, Apple Silicon (MPS) or the CPU. On Colab, a T4 is enough.

## Configuration

Set `LFM3_ADAPTER_ID` to the GRPO adapter you pushed from [`lfm3_vl_grpo.ipynb`](lfm3_vl_grpo.ipynb).

In [ ]:
LFM3_MODEL_ID = "LiquidAI/LFM3-VL-2B"
LFM3_ADAPTER_ID = None  # e.g. "your-username/lfm3-vl-blade-grpo"
SHOW_BASE = True  # also show the base model's answer for each photo
N_PHOTOS = 18  # spread evenly over the 9 defect types, so about two per type

DEVICE = None  # None picks "cuda", then "mps", then "cpu"
DTYPE = None  # None picks "bfloat16" on CUDA and "float32" on MPS/CPU
DATA_DIR = "../data"

PROMPT = (
    "You are inspecting a photo of a wind turbine blade. "
    "Write a brief inspection finding in two or three sentences: "
    "the defect type, the visual evidence that supports it, and where on the blade it is."
)

# Sampling from the LFM3-VL-2B model card.
GENERATION = dict(do_sample=True, temperature=0.1, top_k=50, repetition_penalty=1.1, max_new_tokens=256)

## Setup

`blade_eval.py` holds the dataset loader and the held-out split. On Colab, upload it from the example folder next to this notebook when asked.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:  # locally, `uv run` installs everything from pyproject.toml
    %pip install -q pillow matplotlib "transformers==5.20.0" "peft==0.21.2" "torchao==0.18.0" accelerate

# Locally, blade_eval.py is one folder up. On Colab, upload it next to the notebook.
if os.path.exists("../blade_eval.py"):
    sys.path.insert(0, "..")
elif not os.path.exists("blade_eval.py"):
    raise FileNotFoundError("Upload blade_eval.py from the example folder next to this notebook.")

import textwrap

import matplotlib.pyplot as plt
import torch

import blade_eval as be

assert LFM3_ADAPTER_ID, "Set LFM3_ADAPTER_ID to your GRPO adapter."

## Load the model

The same loader as in `evaluate.ipynb`: the base model with the GRPO adapter on top. For the base model's answers, the adapter is switched off with `disable_adapter()`, so only one copy of the weights is in memory.

In [ ]:
model, processor, device = be.load_local_model(LFM3_MODEL_ID, LFM3_ADAPTER_ID, DEVICE, dtype=DTYPE)
print(f"Loaded {LFM3_MODEL_ID} + {LFM3_ADAPTER_ID} on {device} ({str(model.dtype).removeprefix('torch.')})")

## Pick held-out photos

The photos come from the test split, which neither training notebook uses: about two per defect type.

In [ ]:
root = be.download(DATA_DIR)
items = be.load_items(root, split="test", n=N_PHOTOS, exclude_cross_class_duplicates=True)
items = sorted(items, key=lambda item: be.CLASSES.index(item.label))
print(f"{len(items)} photos:", ", ".join(item.label for item in items))

## Ask for an inspection finding

The prompt asks in plain language and doesn't list the defect types, so any type name in the answer comes from the model. Thinking is off, as in the evaluation.

In [ ]:
@torch.inference_mode()
def finding(item, seed=0):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = processor(text=[prompt], images=item.image(), return_tensors="pt").to(model.device)
    torch.manual_seed(seed)
    output = model.generate(**inputs, **GENERATION)
    return processor.decode(output[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()


answers = []
for item in items:
    fine_tuned = finding(item)
    base = None
    if SHOW_BASE:
        with model.disable_adapter():
            base = finding(item)
    answers.append((item, fine_tuned, base))
    print(f"{item.item_id}: done")

## Look at the answers

Each photo shows the annotated boxes in green and its annotated defect type in the title.

In [ ]:
def wrap(text, width=70):
    return "\n".join(textwrap.fill(line, width) for line in text.splitlines() if line.strip())


for item, fine_tuned, base in answers:
    fig, (image_ax, text_ax) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.3]})
    image_ax.imshow(be.draw(item))
    image_ax.set_title(f"Annotated: {item.label}")
    image_ax.axis("off")
    text = f"Fine-tuned (SFT + GRPO):\n{wrap(fine_tuned)}"
    if base is not None:
        text += f"\n\nBase LFM3-VL-2B:\n{wrap(base)}"
    text_ax.text(0, 1, text, va="top", ha="left", fontsize=10, family="monospace")
    text_ax.axis("off")
    plt.tight_layout()
    plt.show()

## Next steps

- Measure the JSON detections on the full test split with [`evaluate.ipynb`](evaluate.ipynb).
- To grade written findings, you would need reference notes from an inspector. WTBs2025 only has boxes.